# PCA Representation Process

In order to find pair to cluster, we must summarize statistics from large, time-series dataset spread across 7000+ CSV files containing historic information. This project will implement the distance approach where it relies on the price movmeent over a formation peroid (a set period where historical data is analyzed to form pairs). The prices of the stock is normalized so that relative movement (not absolute price) is compared. The euclidean distance between the normalized price series of the stocks is compared (1).

After we collect these key metrics, we will create another CSV file which uses PCA values to cluster to find clusters. This dataset will look like:

| Stock    | Mean(PC1) | Mean(PC2) | Std(PC1) | Std(PC2) |
|----------|-----------|-----------|----------|----------|
| Stock A  | 1.25      | -0.15     | 0.35     | 0.42     |
| Stock B  | -0.90     | 0.80      | 0.29     | 0.50     |
| Stock C  | 0.05      | -0.60     | 0.40     | 0.55     |
| ...      | ...       | ...       | ...      | ...      |

- **Mean(PC1)** and **Mean(PC2)** represent the average values of the first and second principal components for each stock.
- **Std(PC1)** and **Std(PC2)** represent the standard deviation values of the first and second principal components for each stock.

## Expected Inputs
`dataset/stock_csv/`

## Output Files
`summary_statistics_2015_2017.csv`

`pca_data.csv`


In [ ]:
import pandas as pd
import glob
import os

In [ ]:
# Define the directory containing the CSV files
data_dir = 'dataset/stock_csv/'
output_file = 'summary_statistics_2015_2017.csv'

# Define the date range
start_date = '2015-01-01'
end_date = '2017-12-31'

# Define the columns to analyze
columns_to_analyze = [
    'Daily Return',
    'MA-10',
    'EMA',
    'Volatility-10',
    'RSI',
    'Volume MA-20',
    'High-Low',
    'Close-Open',
    'ROC-10',
    'Log Returns'
]

In [ ]:
# Initialize a list to store summary data
summary_data = []

# Get a list of all CSV files in the directory
csv_files = glob.glob(os.path.join(data_dir, '*.csv'))
print(f"Found {len(csv_files)} CSV files in '{data_dir}'.")

for file_path in csv_files:
    try:
        # Extract ticker from filename
        # Example filename: 'AAPL.us.csv'
        base_name = os.path.basename(file_path)
        name_without_ext = os.path.splitext(base_name)[0]  # 'AAPL.us'
        
        # Split at '.us' to get the ticker
        if '.us' in name_without_ext:
            ticker = name_without_ext.split('.us')[0]
        else:
            # If '.us' is not present, use the entire name_without_ext as ticker
            ticker = name_without_ext
        
        print(f"\nProcessing ticker: {ticker} from file: {base_name}")
        
        # Read the CSV file
        df = pd.read_csv(file_path, parse_dates=['Date'])
        total_rows = len(df)
        print(f"Total rows in {ticker}: {total_rows}")
        
        # Filter rows within the date range
        mask = (df['Date'] >= start_date) & (df['Date'] <= end_date)
        df_filtered = df.loc[mask]
        filtered_rows = len(df_filtered)
        print(f"Rows after filtering by date ({start_date} to {end_date}): {filtered_rows}")
        
        if df_filtered.empty:
            print(f"No data for {ticker} in the specified date range. Skipping.")
            continue
        
        # Initialize a dictionary to store statistics for the current ticker
        ticker_stats = {'Ticker': ticker}
        
        for col in columns_to_analyze:
            if col not in df_filtered.columns:
                print(f"Column '{col}' not found in {ticker}. Assigning None for its statistics.")
                ticker_stats[f'{col}_25th'] = None
                ticker_stats[f'{col}_50th'] = None
                ticker_stats[f'{col}_75th'] = None
                continue
            
            # Drop NaN values for accurate statistics
            data = df_filtered[col].dropna()
            
            if data.empty:
                print(f"All values are NaN for column '{col}' in {ticker}. Assigning None for its statistics.")
                ticker_stats[f'{col}_25th'] = None
                ticker_stats[f'{col}_50th'] = None
                ticker_stats[f'{col}_75th'] = None
                continue
            
            # Calculate statistics
            q25 = data.quantile(0.25)
            q50 = data.quantile(0.50)
            q75 = data.quantile(0.75)
            
            ticker_stats[f'{col}_25th'] = q25
            ticker_stats[f'{col}_50th'] = q50
            ticker_stats[f'{col}_75th'] = q75
            
            print(f"{ticker} - {col}: 25th={q25}, 50th={q50}, 75th={q75}")
        
        # Append the ticker's statistics to the summary list
        summary_data.append(ticker_stats)
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# Check if summary_data has any entries
total_processed = len(summary_data)
print(f"\nTotal tickers processed: {total_processed}")

if total_processed == 0:
    print("No data was processed. Please check the date range and data availability.")
else:
    # Convert the summary data to a DataFrame
    summary_df = pd.DataFrame(summary_data)
    
    # Define the desired column order
    ordered_columns = ['Ticker']
    for col in columns_to_analyze:
        ordered_columns.extend([
            f"{col}_25th",
            f"{col}_50th",
            f"{col}_75th"
        ])
    
    # Reorder the DataFrame columns accordingly
    # Handle cases where some columns might not be present by intersecting with existing columns
    ordered_columns = [col for col in ordered_columns if col in summary_df.columns]
    summary_df = summary_df[ordered_columns]
    
    # Save the summary to a CSV file
    summary_df.to_csv(output_file, index=False)
    
    print(f"Summary statistics saved to '{output_file}'.")


## References

1. Smith, R. T., & Xu, X. (2017). A good pair: alternative pairs-trading strategies. *Financial Markets and Portfolio Management, 31*(1), 1-26. doi:10.1007/s11408-016-0280-x

2. Elliott, R. J., Van Der Hoek, J., & Malcolm, W. P. (2005). Pairs trading. *Quantitative Finance, 5*(3), 271-276. doi:10.1080/14697680500149370
